## Building an Object-Detection Training Set from Videos

This demo turns raw videos into a pytorch-ready object-detection training set, using a declarative
Pixeltable schema. The pipeline is a three-level table hierarchy:

```
videos (table)             source videos + metadata
  -> frames (view)         one frame per second (frame_iterator), YOLOX detections as pseudo-labels
  -> training_frames       frames with detections, resized to 640x640, boxes rescaled to match
     (view)
```

Everything downstream of `videos` is computed automatically and incrementally: insert a video, get
training samples.

Requirements: `pip install -U pixeltable pixeltable-yolox torch torchvision`


In [ ]:
# %pip install -qU pixeltable pixeltable-yolox torch torchvision


### The schema

The whole pipeline is declared in [schema.py](./schema.py) as a class-based schema: one class per
table/view, columns as annotations, computed columns as assignments. Two details worth calling out:

- `frame` and the `overlay` columns are **unstored**: they are computed on demand (frames are
  re-extracted from the video when read), so no image data is persisted.
- the box math (`bboxes_resize_canvas`) and visualization (`bboxes_draw`) are Pixeltable built-ins;
  the image and its boxes are transformed together, so they cannot drift apart.


In [ ]:
!pxt schema update schema.py od_demo


In [ ]:
import pixeltable as pxt

videos = pxt.get_table('od_demo.videos')
frames = pxt.get_table('od_demo.frames')
training_frames = pxt.get_table('od_demo.training_frames')


### Insert videos

We use a handful of public videos from the
[Multimedia Commons](http://mmcommons.org/) S3 bucket (no credentials needed). Inserting them runs the
entire pipeline: frame extraction, YOLOX inference, and training-sample construction.


In [ ]:
S3_PREFIX = 's3://multimedia-commons/data/videos/mp4/'
video_paths = [
    '9cd/047/9cd047335f12962a5e633c3b9e6602e.mp4',
    '111/015/1110153dc81d833025828f875b3ad2.mp4',
    '888/061/8880616a157cfd9036fc16ad6cf5c3af.mp4',
    'abc/01a/abc01aa533364ca1597e0f287f393b9.mp4',
    '00a/a5b/00aa5b8cc17f67eded6a257a61119.mp4',
    '7ab/023/7ab02326a7c9ab4728681f63aaa7c36d.mp4',
]
videos.insert([{"video": S3_PREFIX + p} for p in video_paths])
print(f'{videos.count()} videos -> {frames.count()} frames -> {training_frames.count()} training samples')


### Inspect the pseudo-labels

The `overlay` column renders the YOLOX detections on the frame. It is unstored: the image below is
produced at query time (the same column can be browsed in the dashboard).


In [ ]:
frames.select(frames.overlay, frames.detections).show(2)


### Incremental computation

Insert one more video: only its frames are processed, everything already computed stays untouched.


In [ ]:
videos.insert([{'video': S3_PREFIX + 'cde/078/cde078a6803a16cae79eb438e178f0ce.mp4'}])
print(f'{videos.count()} videos -> {frames.count()} frames -> {training_frames.count()} training samples')


### Image transforms for detection training

Object-detection training uses two kinds of transforms, and they belong in different places:

| | examples | where |
|---|---|---|
| **deterministic preprocessing** | decode, resize/letterbox to model input size, rescale boxes, drop degenerate boxes, coordinate-format conversion | Pixeltable computed columns: computed once, cached, versioned, shared |
| **stochastic augmentation** | random flip, scale jitter, random crop, mosaic/mixup, color jitter, random erasing | the training DataLoader: must differ every epoch, so it is never materialized |

Geometric augmentations must transform the boxes along with the image - use
`torchvision.transforms.v2` with `tv_tensors.BoundingBoxes` (or albumentations with `bbox_params`).

The `training_frames` view holds the deterministic part: 640x640 images with boxes rescaled to match.
Its `overlay` column lets us verify the two stayed consistent:


In [ ]:
training_frames.select(training_frames.overlay, training_frames.labels).show(2)


### Export to pytorch

`to_pytorch_dataset()` serializes the query result to a disk cache once and returns an
`IterableDataset`; re-running epochs reads from the cache instead of re-executing the query.
With `image_format='pt'`, images arrive as CxHxW float32 tensors in [0, 1].

The custom `collate_fn` is needed because each sample has a different number of boxes; it is also the
natural home for the stochastic augmentations.


In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import tv_tensors
from torchvision.transforms import v2

dataset = training_frames.select(
    training_frames.image, training_frames.boxes, training_frames.labels
).to_pytorch_dataset(image_format='pt')

augment = v2.RandomHorizontalFlip(p=0.5)

def collate(samples: list[dict]) -> dict:
    images, boxes, labels = [], [], []
    for s in samples:
        bb = tv_tensors.BoundingBoxes(
            torch.tensor(s['boxes'], dtype=torch.float32), format='XYXY', canvas_size=(640, 640)
        )
        img, bb = augment(s['image'], bb)
        images.append(img)
        boxes.append(torch.as_tensor(bb))
        labels.append(torch.tensor(s['labels'], dtype=torch.int64))
    return {'images': torch.stack(images), 'boxes': boxes, 'labels': labels}

loader = DataLoader(dataset, batch_size=8, collate_fn=collate)
batch = next(iter(loader))
print('images:', batch['images'].shape, batch['images'].dtype)
print('boxes[0]: ', batch['boxes'][0].shape)
print('labels[0]:', batch['labels'][0].shape)


### Where to go from here

- `to_coco_dataset()` exports the same data as a COCO json for other tooling.
- An embedding index on `frames.frame` turns curation into a query: find more frames like a hard example.
- Label with a larger model (`yolox_x`) and evaluate a smaller one against it with
  `pxtf.vision.eval_detections()` + `mean_ap()` - a two-computed-column eval harness.
